# Mean-variance portfolio

Classic Markowitz: maximize $\mu^\top w - \tfrac{\gamma}{2}\, w^\top \Sigma w$ with $\mu$ = mean daily returns, $\Sigma$ = return covariance. We first write the unconstrained closed form, then solve the **long-only** portfolio ($w \ge 0$, $\sum w = 1$) by parametrizing $w = \mathrm{softmax}(z)$ and doing gradient ascent on $z$ — the constraints hold by construction.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

from stonks import get_prices, to_returns

%matplotlib inline


## Parameters


In [ ]:
PERIOD = "2y"
INTERVAL = "1d"
TOP_N = 50
FIELD = "close"   # adjusted close -> total returns
gamma = 2.0       # risk aversion (your objective mu^T w - w^T Sigma w is gamma = 2)


## Fetch prices and compute returns


In [ ]:
prices = get_prices(top_n=TOP_N, period=PERIOD, interval=INTERVAL, field=FIELD)
returns = to_returns(prices).dropna()   # N tickers x (T-1) daily simple returns
print("prices:", prices.shape, "| returns:", returns.shape)


## Expected returns $\mu$ and covariance $\Sigma$

$\Sigma$ is built explicitly as the stock×stock covariance (`returns.cov()` would instead return a dates×dates matrix, since pandas treats columns as variables).


In [ ]:
X = returns.to_numpy(dtype=float)
tickers = list(returns.index)
mu = X.mean(axis=1)
Xc = X - mu[:, None]
Sigma = (Xc @ Xc.T) / (Xc.shape[1] - 1)   # N x N sample covariance
print("mu:", mu.shape, "| Sigma:", Sigma.shape, "| cond(Sigma): %.1e" % np.linalg.cond(Sigma))


## Unconstrained closed form

Setting $\nabla=0$ in $\mu^\top w - \tfrac{\gamma}{2}w^\top\Sigma w$ gives $w^* = \tfrac{1}{\gamma}\Sigma^{-1}\mu$. This is the literal argmax of your objective, but it is **not a portfolio** — no normalization, extreme long/short weights. (Works only while $T > N$, so $\Sigma$ is invertible.)


In [ ]:
w_unc = np.linalg.solve(Sigma, mu) / gamma
print("unconstrained: sum(w)=%.3f  min=%.3f  max=%.3f" % (w_unc.sum(), w_unc.min(), w_unc.max()))


## Long-only portfolio via softmax

To get a usable portfolio with $w \ge 0$ and $\sum_i w_i = 1$, parametrize $w = \mathrm{softmax}(z)$ for a free vector $z$. The constraints are then satisfied **by construction**, and we do unconstrained gradient ascent on $z$ (torch autograd). This matches the true QP optimum to numerical precision.


In [ ]:
mu_t = torch.tensor(mu, dtype=torch.float64)
Sig_t = torch.tensor(Sigma, dtype=torch.float64)
n = len(mu)

z = torch.zeros(n, dtype=torch.float64, requires_grad=True)   # free parameters
opt = torch.optim.Adam([z], lr=0.5)
for _ in range(8000):
    opt.zero_grad()
    w = torch.softmax(z, dim=0)                              # w >= 0, sum(w) = 1 by construction
    loss = -(mu_t @ w - 0.5 * gamma * w @ Sig_t @ w)          # maximize f -> minimize -f
    loss.backward()
    opt.step()

w_long = torch.softmax(z, dim=0).detach().numpy()
print("sum(w)=%.6f  min(w)=%.2e  holdings(>1e-3)=%d / %d"
      % (w_long.sum(), w_long.min(), int((w_long > 1e-3).sum()), n))


## Result


In [ ]:
ann_ret = w_long @ mu * 252
ann_vol = np.sqrt(w_long @ Sigma @ w_long) * np.sqrt(252)
print("long-only annualized: return=%.1f%%  vol=%.1f%%  Sharpe=%.2f"
      % (ann_ret * 100, ann_vol * 100, ann_ret / ann_vol))

weights_long = pd.Series(w_long, index=tickers).sort_values(ascending=False)
print("\nholdings:")
print(weights_long[weights_long > 1e-3].round(3))


In [ ]:
nz = weights_long[weights_long > 1e-3]
fig, ax = plt.subplots(figsize=(9, 5))
ax.bar(range(len(nz)), nz.values)
ax.set_xticks(range(len(nz)))
ax.set_xticklabels(nz.index, rotation=90, fontsize=8)
ax.set_ylabel("weight")
ax.set_title("Long-only Markowitz weights ($w = \mathrm{softmax}(z)$, $\sum w = 1$)")


## Notes

- **Constraints for free.** $w=\mathrm{softmax}(z)$ enforces $w\ge0$ and $\sum w=1$ structurally, so the outer problem is unconstrained ascent on $z$ — no projection or QP solver needed. (Verified to reach the true long-only QP optimum to ~6 significant figures.)
- **$\Sigma$ invertible here** because $T>N$ (501 days vs 45 stocks). For the 1500-stock universe ($N>T$) the sample covariance is singular — use shrinkage / a factor model first.
- **Still in-sample.** Both $\mu$ and $\Sigma$ are estimated from the same window, so the Sharpe is optimistic; $\mu$ in particular is noisy, which drives the concentration. Raise $\gamma$ to diversify, or shrink $\mu$ (Black-Litterman) for stabler weights.
